# Module 12: Building a Peer Benchmark Series

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every module so far has worked on one agency's series. None of them can tell
you whether what happened to that agency happened everywhere.

This notebook builds the second line on the chart: a monthly benchmark series
constructed from the agencies most like the one you are studying. It is the
last piece [Module 16](Module_16_Did_Something_Change.ipynb) needs, and the
whole causal inference series is built on it.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

profile = pd.read_csv(BASE + "agency_profile.csv")
profile["short"] = (profile["agency_name"]
                    .str.replace(" Police Department", "", regex=False)
                    .str.replace(" Sheriff's Office", "", regex=False)
                    .str.replace(" Police", "", regex=False))

TREATED = ["A001", "A002", "A004", "A007", "A010"]   # adopted the training
PROGRAM_START = "2023-07"

profile[["agency_id", "short", "agency_type", "sworn_officers",
         "population_served", "region"]]

## 2. Gower distance

Agency characteristics mix **numbers** with **labels**, and ordinary distance
measures cannot handle the mixture. Gower distance can: each variable
contributes something between 0 and 1, and the contributions are averaged.

- a **number**: the absolute difference divided by that variable's range
- a **label**: 0 if the two agencies match, 1 if they do not

In [ ]:
NUMERIC = ["sworn_officers", "population_served", "violent_crime_rate_per_1000",
           "property_crime_rate_per_1000", "budget_share_public_safety_pct",
           "county_population"]
CATEGORICAL = ["agency_type", "region"]
LOG_SCALE = ["sworn_officers", "population_served", "county_population"]


def gower(df, numeric=NUMERIC, categorical=CATEGORICAL, log_scale=LOG_SCALE):
    """Distance between every pair of rows, mixing numbers and labels."""
    d = df.copy()
    for c in log_scale:
        d[c] = np.log(d[c])

    n = len(d)
    total = np.zeros((n, n))
    for c in numeric:
        v = d[c].astype(float).values
        total += np.abs(v[:, None] - v[None, :]) / (v.max() - v.min())
    for c in categorical:
        v = d[c].values
        total += (v[:, None] != v[None, :]).astype(float)
    return total / (len(numeric) + len(categorical))

`log_scale` is a judgment, not a technicality. Agency size runs from 8 officers
to 902, so on a raw scale every small agency looks equally close to every other
and the largest sits alone. Taking logs makes the distance reflect
**proportional** difference, which is how people think about agency size, and
it changes who ends up in which group.

In [ ]:
D = gower(profile)
distance = pd.DataFrame(D, index=profile["agency_id"], columns=profile["agency_id"])
label = dict(zip(profile["agency_id"], profile["short"]))

for aid in profile["agency_id"]:
    near = distance.loc[aid].drop(aid).nsmallest(3)
    print(f"{label[aid]:28s} " + ",  ".join(f"{label[k]} ({v:.2f})"
                                            for k, v in near.items()))

Sensible groupings: Millgate with Kelsmoor at 0.09, Summit County with
Lakeshore County at 0.12. Two agencies have no close peer at all.

In [ ]:
loneliness = pd.Series({label[a]: distance.loc[a].drop(a).nsmallest(3).mean()
                        for a in distance.index}).sort_values(ascending=False)
print("mean distance to the three nearest agencies")
print(loneliness.round(2).to_string())

Ashfell is the largest agency in the state and Pinecrest is the only campus
force, so neither has a real peer group. That is a finding to report, not a
problem to work around: for those two the honest comparison is the statewide
figure, and the report should say why.

## 3. Turning a peer group into a series

Now the part that makes this a time series module. A peer group is only useful
once it becomes a **line on the chart**, month by month, for as long as the
data runs.

**Pool the counts and the denominators. Do not average the peers' rates.**

In [ ]:
clean = monthly[monthly["provisional"] == 0].copy()
clean = clean[~((clean["agency_id"] == "A002") &
                (clean["year_month"] == "2021-06"))]      # documented unrest


def benchmark(agency_ids, window=12):
    """Pooled monthly rate for a group of agencies, twelve month trailing."""
    w = (clean[clean["agency_id"].isin(agency_ids)]
         .groupby("year_month")[["n_uof", "n_arrests"]].sum())
    r = 100 * w["n_uof"].rolling(window).sum() / w["n_arrests"].rolling(window).sum()
    r.index = pd.PeriodIndex(w.index, freq="M").to_timestamp()
    return r


def peers_of(aid, k=3, exclude_treated=False):
    order = distance.loc[aid].drop(aid).sort_values()
    if exclude_treated:
        order = order[~order.index.isin(TREATED)]
    return list(order.index[:k])

### Why pooling rather than averaging

In [ ]:
peers = peers_of("A001", exclude_treated=True)
w = clean[clean["agency_id"].isin(peers)]

each = w.groupby("agency_id").apply(lambda d: 100 * d["n_uof"].sum() / d["n_arrests"].sum())
share = w.groupby("agency_id")["n_arrests"].sum() / w["n_arrests"].sum()

print("peers:", [label[i] for i in peers])
print("\nrate of each peer  :", {label[k]: round(v, 2) for k, v in each.items()})
print("share of arrests   :", {label[k]: round(v, 3) for k, v in share.items()})
print(f"\nsimple average of the three rates : {each.mean():.3f}")
print(f"pooled counts over pooled arrests : "
      f"{100 * w['n_uof'].sum() / w['n_arrests'].sum():.3f}")

Ashfell supplies about 82 percent of the group's arrests. Pooling gives it 82
percent of the weight; averaging gives it a third, which lets two agencies with
a tenth of the activity move the benchmark as much as the one with most of it.

Pooling answers "what rate did a person arrested by one of these agencies
face". Averaging answers "what is the mean of three agency level numbers".
The first is almost always the question.

## 4. The trap: peers who did the same thing

Stonewick adopted the de escalation training in July 2023. Look at who its
nearest peers are.

In [ ]:
near3 = peers_of("A001")
for i in near3:
    flag = "  <-- also adopted the training" if i in TREATED else ""
    print(f"  {label[i]:16s} distance {distance.loc['A001', i]:.2f}{flag}")

Two of the three. And this is not bad luck with one dataset: agencies adopt a
programme because of what they are, so a peer group built on what they are will
be enriched with agencies that made the same decision. **The more carefully the
peer group is matched, the more likely this becomes.**

In [ ]:
untreated3 = peers_of("A001", exclude_treated=True)
print("nearest three, no screening:", [label[i] for i in near3])
print("nearest three, screened    :", [label[i] for i in untreated3])

## 5. What the wrong comparison group costs

In [ ]:
PRE = ("2021-07", "2023-06")        # two years before
POST = ("2023-11", "2026-04")       # from the month the programme was fully in place


def pooled(ids, lo, hi):
    w = clean[(clean["agency_id"].isin(ids)) &
              (clean["year_month"] >= lo) & (clean["year_month"] <= hi)]
    return 100 * w["n_uof"].sum() / w["n_arrests"].sum()


riverbend_before, riverbend_after = pooled(["A001"], *PRE), pooled(["A001"], *POST)

rows = []
for name, ids in [("three nearest peers, no screening", near3),
                  ("three nearest peers, screened", untreated3),
                  ("every agency that was not trained",
                   [a for a in distance.index if a not in TREATED])]:
    before, after = pooled(ids, *PRE), pooled(ids, *POST)
    rows.append({
        "comparison group": name,
        "benchmark before": round(before, 2),
        "benchmark after": round(after, 2),
        "estimated effect, percent":
            round(100 * ((riverbend_after / riverbend_before) / (after / before) - 1), 1),
    })

print(f"Stonewick itself: {riverbend_before:.2f} before, {riverbend_after:.2f} after\n")
pd.DataFrame(rows).set_index("comparison group")

**With the trained peers left in, the programme disappears.** The benchmark
fell for the same reason Stonewick fell, so the comparison subtracts the effect
from itself. Had all three peers been treated, the estimate could have come out
positive and a report would have concluded the training made things worse.

Screening the peers recovers about −7 percent against a built in truth of
**−12 percent**. That is the right method and it is still short, because this
is one agency over thirty months. The multi agency estimate in
[GROUND_TRUTH.md](../../../Data/GROUND_TRUTH.md) recovers −12.6 percent using
every trained agency at once.

**A single agency does not carry enough information to measure an effect this
size**, which is [Module 4](Module_04_Why_Small_Agencies_Look_Volatile.ipynb)
appearing yet again.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(benchmark(["A001"]), color="#eb6834", lw=2.4, label="Stonewick")
ax.plot(benchmark(near3), color="#8a8880", lw=2,
        label="three nearest peers, two of them also trained")
ax.plot(benchmark(untreated3), color="#2a78d6", lw=2,
        label="three nearest peers, screened")
ax.axvline(pd.Timestamp(PROGRAM_START + "-01"), color="k", ls="--", lw=1.1)
ax.set_ylabel("use of force per 100 arrests, twelve month trailing")
ax.legend(fontsize=8, frameon=False, loc="lower left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## 6. What to carry away

| Habit | Why |
|---|---|
| Build peer groups from several characteristics | size alone matches agencies with nothing in common |
| Use Gower distance | agency data is always numbers plus labels |
| Take logs of size variables | difference in size is proportional, not absolute |
| Pool counts, do not average rates | the largest peer should carry the most weight |
| **Screen peers for the intervention** | **otherwise the effect cancels itself out** |
| Report how far away the peers are | a weak peer group makes the comparison meaningless |
| Build the whole series, not one year | a benchmark from one year cannot see a change |
| Let agencies review their own peer group | a comparison nobody accepts changes nothing |

## Exercise

Drop `agency_type` from the distance and rebuild the peer groups. Who moves,
and what does that tell you about what the variable was doing?

In [ ]:
# Fill in the blank, then run.
CATEGORICAL_TO_USE = None       # try ["region"]

if CATEGORICAL_TO_USE is not None:
    D2 = gower(profile, categorical=CATEGORICAL_TO_USE)
    d2 = pd.DataFrame(D2, index=profile["agency_id"], columns=profile["agency_id"])
    for aid in ["A010", "A011", "A007"]:
        before = [label[i] for i in distance.loc[aid].drop(aid).nsmallest(3).index]
        after = [label[i] for i in d2.loc[aid].drop(aid).nsmallest(3).index]
        print(f"{label[aid]}\n   with agency type:    {before}\n"
              f"   without agency type: {after}\n")
else:
    print("Set CATEGORICAL_TO_USE above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
CATEGORICAL_TO_USE = ["region"]
```

Pinecrest and Dunmoor Tribal change peers; the sheriff's offices change
less. Without `agency_type`, a campus force and a tribal department become
matchable with any municipal department of similar size, because nothing else
in the variable list captures the difference in what those agencies are for.

The lesson is about variable choice, not about method. Gower distance will
faithfully average whatever you hand it, and a characteristic left out of the
list is a characteristic the peer groups will ignore. Deciding what belongs in
the list is a domain judgment, which is why the WADEPS framework validates its
peer groups with people who know the agencies rather than with a distance
measure alone.

</details>

---

**Next:** Part IV of the Intermediate series, on forecasting and change
detection: the baseline forecasts you have to beat, exponential smoothing,
measuring forecast error, and telling whether something changed.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*